In [ ]:
# Uninstall old stack remnants from the SDXL/IP-Adapter approach
!pip uninstall -y diffusers transformers accelerate huggingface-hub pillow

!pip install -q diffusers transformers accelerate pillow==10.2.0 huggingface-hub
!pip install -q opencv-python insightface onnxruntime-gpu controlnet_aux



In [2]:
!git clone https://github.com/Zheng-Chong/CatVTON.git /kaggle/working/CatVTON
!git clone https://github.com/instantX-research/InstantID.git /kaggle/working/InstantID

fatal: destination path '/kaggle/working/CatVTON' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/InstantID' already exists and is not an empty directory.


In [3]:
import sys
sys.path.insert(0, "/kaggle/working/CatVTON")

In [4]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("ngrok_token")

In [5]:
import os
target_dir = "/kaggle/working/InstantID/models/antelopev2"
os.makedirs(target_dir, exist_ok=True)

from huggingface_hub import hf_hub_download

antelope_files = [
    "1k3d68.onnx",
    "2d106det.onnx",
    "genderage.onnx",
    "glintr100.onnx",
    "scrfd_10g_bnkps.onnx",
]

for fname in antelope_files:
    hf_hub_download(
        repo_id="DIAMONIK7777/antelopev2",
        filename=fname,
        local_dir=target_dir
    )

onnx_files = [f for f in os.listdir(target_dir) if f.endswith(".onnx")]
print("antelopev2 models found:", onnx_files)
assert len(onnx_files) == 5, f"Expected 5 .onnx files, found {len(onnx_files)} — download likely incomplete"

1k3d68.onnx: reconstructing file:   0%|          |  0.00B /  144MB            

1k3d68.onnx: downloading bytes:           |  0.00B            

2d106det.onnx: reconstructing file:   0%|          |  0.00B / 5.03MB            

2d106det.onnx: downloading bytes:           |  0.00B            

genderage.onnx: reconstructing file:   0%|          |  0.00B / 1.32MB            

...

glintr100.onnx: reconstructing file:   0%|          |  0.00B /  261MB            

glintr100.onnx: downloading bytes:           |  0.00B            

scrfd_10g_bnkps.onnx: reconstructing file:   0%|          |  0.00B / 16.9MB            

scrfd_10g_bnkps.onnx: downloading bytes:           |  0.00B            

antelopev2 models found: ['glintr100.onnx', 'scrfd_10g_bnkps.onnx', 'genderage.onnx', '1k3d68.onnx', '2d106det.onnx']


In [6]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("/kaggle/working/InstantID/checkpoints", exist_ok=True)

hf_hub_download(repo_id="InstantX/InstantID", filename="ControlNetModel/config.json",
                 local_dir="/kaggle/working/InstantID/checkpoints")
hf_hub_download(repo_id="InstantX/InstantID", filename="ControlNetModel/diffusion_pytorch_model.safetensors",
                 local_dir="/kaggle/working/InstantID/checkpoints")
hf_hub_download(repo_id="InstantX/InstantID", filename="ip-adapter.bin",
                 local_dir="/kaggle/working/InstantID/checkpoints")

assert os.path.exists("/kaggle/working/InstantID/checkpoints/ControlNetModel/config.json")
print("InstantID checkpoints ready:", os.listdir("/kaggle/working/InstantID/checkpoints"))

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

ControlNetModel/diffusion_pytorch_model.(…): reconstructing file:   0%|          |  0.00B / 2.50GB            

ControlNetModel/diffusion_pytorch_model.(…): downloading bytes:           |  0.00B            

ip-adapter.bin: reconstructing file:   0%|          |  0.00B / 1.69GB            

ip-adapter.bin: downloading bytes:           |  0.00B            

InstantID checkpoints ready: ['ControlNetModel', 'ip-adapter.bin', '.cache']


In [7]:
import os
print("InstantID dir exists:", os.path.isdir("/kaggle/working/InstantID"))
if os.path.isdir("/kaggle/working/InstantID"):
    print(os.listdir("/kaggle/working/InstantID"))

InstantID dir exists: True
['.github', 'cog.yaml', 'README.md', 'checkpoints', 'pipeline_stable_diffusion_xl_instantid_img2img.py', 'gradio_demo', '.gitignore', 'pipeline_stable_diffusion_xl_instantid_full.py', 'LICENSE', 'models', 'docs', 'examples', '.git', 'infer_img2img.py', 'ip_adapter', 'pipeline_stable_diffusion_xl_instantid.py', 'cog', 'assets', 'infer.py', 'infer_full.py']


In [8]:
import sys
sys.path.insert(0, "/kaggle/working/InstantID")

import torch, gc, cv2
import numpy as np
from PIL import Image
from insightface.app import FaceAnalysis
from diffusers.models import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps

class AvatarEngine:
    def __init__(self):
        print("Initializing Avatar Generator (InstantID)...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.face_app = FaceAnalysis(
            name="antelopev2", root="/kaggle/working/InstantID",
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
        )
        self.face_app.prepare(ctx_id=0, det_size=(640, 640))

        controlnet = ControlNetModel.from_pretrained(
            "/kaggle/working/InstantID/checkpoints/ControlNetModel",
            torch_dtype=torch.float16
        )
        self.pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            controlnet=controlnet, torch_dtype=torch.float16
        )
        self.pipe.load_ip_adapter_instantid("/kaggle/working/InstantID/checkpoints/ip-adapter.bin")
        self.pipe.enable_model_cpu_offload()
        print("Avatar Generator ready!")

    def generate_avatar(self, user_image_path, output_path):
        face_image = Image.open(user_image_path).convert("RGB")
        faces = self.face_app.get(cv2.cvtColor(np.array(face_image), cv2.COLOR_RGB2BGR))
        if not faces:
            raise ValueError("No face detected in uploaded photo.")

        face_info = sorted(faces, key=lambda f: (f["bbox"][2]-f["bbox"][0])*(f["bbox"][3]-f["bbox"][1]))[-1]
        face_emb = face_info["embedding"]
        face_kps = draw_kps(face_image, face_info["kps"])

        # Strictly enforce direct eye-contact and forward catalog orientation
        prompt = (
            "full body studio portrait of a male fashion model standing facing front, "
            "looking directly at the camera, head centered, straight frontal view, "
            "relaxed A-pose with arms held slightly away from sides, palms visible, "
            "wearing a plain fitted white t-shirt and plain neutral trousers, "
            "seamless grey studio backdrop, bright symmetrical studio fashion lighting, 8k, sharp focus"
        )
        
        # Eliminate profile, turned, side, and occluded head orientations
        negative_prompt = (
            "side view, profile view, looking sideways, head turned, sunglasses, dark glasses, eyewear, "
            "turned away, 3/4 view, walking, motion, back view, bent arms, hands in pockets, crossed legs, "
            "blurry, cropped head, asymmetric posture, high contrast shadows"
        )

        # Fix the seed for deterministic frontal generation
        generator = torch.Generator(device="cuda").manual_seed(2026)

        result = self.pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image_embeds=face_emb,
            image=face_kps,
            controlnet_conditioning_scale=0.85,
            ip_adapter_scale=0.8,
            num_inference_steps=30,
            generator=generator,
            height=1024, 
            width=768,
        ).images[0]

        result.save(output_path)
        torch.cuda.empty_cache()
        gc.collect()
        return output_path

In [9]:
!rm -rf /kaggle/working/cached_avatar_*.png

In [10]:
import cv2
import numpy as np
import torch
from PIL import Image
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

class HumanParsingEngine:
    def __init__(self):
        print("Initializing Anatomical Human Parser...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
        self.model = SegformerForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to(self.device)
        print("Human Parser ready!")

    def generate_agnostic_mask(self, avatar_image_path: str, category: str, output_path: str):
        image = Image.open(avatar_image_path).convert("RGB")
        
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
            
        logits = outputs.logits.cpu()
        upsampled_logits = torch.nn.functional.interpolate(
            logits, 
            size=image.size[::-1], 
            mode="bilinear", 
            align_corners=False
        )
        parse_map = upsampled_logits.argmax(dim=1)[0].numpy()
        
        mask = np.zeros_like(parse_map, dtype=np.uint8)

        # Map semantic categories to ATR anatomical classes
        # ATR Classes: 4=Upper-clothes, 6=Pants, 12=Left-leg, 13=Right-leg, 14=Left-arm, 15=Right-arm
        category = category.lower()
        if category in ["top", "outerwear", "shirt", "t-shirt"]:
            mask[parse_map == 4] = 255   # Current Upper-clothes
            mask[parse_map == 14] = 255  # Left-arm
            mask[parse_map == 15] = 255  # Right-arm
        elif category in ["bottom", "pants", "shorts", "jeans"]:
            mask[parse_map == 6] = 255   # Current Pants
            mask[parse_map == 12] = 255  # Left-leg
            mask[parse_map == 13] = 255  # Right-leg
        elif category in ["dress", "jumpsuit"]:
            mask[parse_map == 4] = 255
            mask[parse_map == 6] = 255
            mask[parse_map == 14] = 255
            mask[parse_map == 15] = 255
            mask[parse_map == 12] = 255
            mask[parse_map == 13] = 255

        # Dilate mask to cover the old garment's seams and prevent white fringe artifacts
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
        dilated_mask = cv2.dilate(mask, kernel, iterations=2)

        Image.fromarray(dilated_mask).convert("L").save(output_path)
        return output_path

# Initialize globally alongside your other engines
parser_engine = HumanParsingEngine()

Initializing Anatomical Human Parser...


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  109MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Human Parser ready!


In [11]:
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import json
import re

# 1. Load the lightweight VLM onto Kaggle's GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen2-VL-2B-Instruct"

print("Loading Wardrobe Vision Model...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

def digitize_clothing_item(image_path: str) -> dict:
    """
    Analyzes an image of a clothing item and returns structured metadata.
    """
    image = Image.open(image_path).convert("RGB")
    
    # Define a strict prompt to ensure we get clean JSON back without regular prose text
    prompt = (
        "Analyze this clothing item. Return ONLY a raw JSON object with these keys: "
        "'category' (e.g., top, bottom, shoes, outerwear), 'color', 'style' (e.g., casual, streetwear, formal, vintage), "
        "and 'description' (a brief visual summary). Do not include markdown formatting or backticks."
    )
    
    # Format the multimodal input
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Preprocess and run inference
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=150)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        global output_text
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    
    # Parse output string into a Python dictionary
    try:
        # STRIP MARKDOWN: Remove ```json and ``` fences before passing to json.loads()
        cleaned_text = re.sub(r'```(?:json)?|```', '', output_text).strip()
        metadata = json.loads(cleaned_text)
        return metadata
    except json.JSONDecodeError:
        print("Failed to parse JSON. Raw output:", output_text)
        return {"category": "unknown", "color": "unknown", "style": "unknown", "description": output_text}


# --- QUICK TEST ---
# To test this in Kaggle, upload a picture of a shirt/shoe as 'test_item.jpg' and run:
# print(digitize_clothing_item("test_item.jpg"))

Loading Wardrobe Vision Model...


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [12]:
print(digitize_clothing_item("/kaggle/input/datasets/favourachara/test-images/IMG-20260628-WA0137(1).jpg"))

{'category': 'outerwear', 'color': 'red', 'style': 'formal', 'description': 'A man wearing a traditional red outfit with intricate embroidery, standing in front of a crowd.'}


In [13]:
import json
import re

def clean_and_parse_json(raw_output: str):
    """Strips markdown formatting and parses the JSON string."""
    # Remove the ```json and ``` markers using regex
    cleaned_text = re.sub(r'```(?:json)?|```', '', output_text).strip()
    
    try:
        parsed_dict = json.loads(cleaned_text)
        return parsed_dict
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        # Fallback if it completely fails
        return {'category': 'unknown', 'color': 'unknown', 'style': 'unknown', 'description': cleaned_text}


my_digitized_shirt = clean_and_parse_json(output_text)
print(my_digitized_shirt)

{'category': 'outerwear', 'color': 'red', 'style': 'formal', 'description': 'A man wearing a traditional red outfit with intricate embroidery, standing in front of a crowd.'}


In [14]:
import torch
import gc
from PIL import Image
from model.pipeline import CatVTONPipeline

class CatVTONEngine:
    def __init__(self):
        print("Initializing CatVTON Pipeline...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.pipeline = CatVTONPipeline(
            base_ckpt="booksforcharlie/stable-diffusion-inpainting",
            attn_ckpt="zhengchong/CatVTON",
            attn_ckpt_version="mix",
            weight_dtype=torch.float16,
            use_tf32=True,
            device=self.device,
        )
        print("CatVTON ready!")

    # 1. Add req_id to the parameters
    def generate_tryon_fit(self, user_image_path, mask_image_path, garment_image_path, req_id):
        person_img = Image.open(user_image_path).convert("RGB").resize((768, 1024))
        mask_img = Image.open(mask_image_path).convert("L").resize((768, 1024))
        garment_img = Image.open(garment_image_path).convert("RGB").resize((768, 1024))

        print("Running CatVTON inference...")
        result = self.pipeline(
            image=person_img,
            condition_image=garment_img,
            mask=mask_img,
            num_inference_steps=50,
            guidance_scale=2.5,
        )[0]
        
        # 2. Inject req_id into the dynamic output path
        output_path = f"/kaggle/working/final_{req_id}.png"
        
        result.save(output_path)
        print(f"Success! VTON saved to {output_path}")

        torch.cuda.empty_cache()
        gc.collect()
        return output_path

In [15]:
!pip install pyngrok


In [ ]:
import os
import sys
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from kaggle_secrets import UserSecretsClient
import uuid
import traceback
from fastapi import HTTPException

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_credentials=True,
    allow_methods=['*'], allow_headers=['*'],
)

avatar_engine = AvatarEngine()   # from cell 7
vton_engine = CatVTONEngine()    # from your original CatVTON cell (the one that clones into /kaggle/working/CatVTON)

@app.post("/digitize")
async def digitize(user_image: UploadFile = File(...)):
    image_path = f"/kaggle/working/temp_{user_image.filename}"
    with open(image_path, "wb") as f:
        f.write(await user_image.read())
    metadata = digitize_clothing_item(image_path)
    return {"metadata": metadata}

@app.post("/try-on")
async def try_on(
    user_avatar: UploadFile = File(...),
    top_garment: UploadFile = File(...),
    bottom_garment: UploadFile = File(None)  # Optional bottom garment
):
    try:
        req_id = uuid.uuid4().hex
        
        # 1. Manage Cached Avatar
        safe_avatar_name = "".join(c for c in user_avatar.filename if c.isalnum() or c in "._-")
        avatar_path = f"/kaggle/working/cached_avatar_{safe_avatar_name}.png"
        
        if not os.path.exists(avatar_path):
            upload_temp = f"/kaggle/working/upload_{req_id}.png"
            with open(upload_temp, "wb") as f:
                f.write(await user_avatar.read())
            avatar_engine.generate_avatar(upload_temp, avatar_path)
            if os.path.exists(upload_temp):
                os.remove(upload_temp)
        
        current_canvas = avatar_path
        
        # 2. Inpaint Top Garment
        top_path = f"/kaggle/working/top_{req_id}.png"
        with open(top_path, "wb") as f:
            f.write(await top_garment.read())
            
        mask_top = f"/kaggle/working/mask_top_{req_id}.png"
        parser_engine.generate_agnostic_mask(current_canvas, category="top", output_path=mask_top)
        
        # Render top
        current_canvas = vton_engine.generate_tryon_fit(current_canvas, mask_top, top_path, f"{req_id}_top")

        # 3. Inpaint Bottom Garment (if provided)
        if bottom_garment is not None:
            bottom_path = f"/kaggle/working/bottom_{req_id}.png"
            with open(bottom_path, "wb") as f:
                f.write(await bottom_garment.read())
                
            mask_bottom = f"/kaggle/working/mask_bottom_{req_id}.png"
            parser_engine.generate_agnostic_mask(current_canvas, category="bottom", output_path=mask_bottom)
            
            # Render bottom over the canvas that already has the new top
            current_canvas = vton_engine.generate_tryon_fit(current_canvas, mask_bottom, bottom_path, f"{req_id}_full")

        return FileResponse(current_canvas)

    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))
        
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("ngrok_token")   # matches the key name in cell 3 — note lowercase, cell 14 used "NGROK_TOKEN" which may not match what you actually named the secret
ngrok.set_auth_token(ngrok_token)

ngrok_tunnel = ngrok.connect(8000)
print('>>> KAGGLE GPU API URL:', ngrok_tunnel.public_url)

nest_asyncio.apply()
config = uvicorn.Config(app, port=8000)
server = uvicorn.Server(config)
await server.serve()

Initializing Avatar Generator (InstantID)...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

...

100%|██████████| 50/50 [02:01<00:00,  2.44s/it]


Success! VTON saved to /kaggle/working/final_3c54fd3368d545199c7922e36e16ed11_top.png
Running CatVTON inference...


100%|██████████| 50/50 [02:01<00:00,  2.43s/it]


Success! VTON saved to /kaggle/working/final_3c54fd3368d545199c7922e36e16ed11_full.png
INFO:     102.91.135.150:0 - "POST /try-on HTTP/1.1" 200 OK
Running CatVTON inference...


 80%|████████  | 40/50 [01:37<00:24,  2.42s/it]